# Stage B v5 — Single-query proof of concept on val_009

**Goal**: lift val_009 F1 from 0.000 (v4) to something meaningful by giving the LLM the per-aspect Swiss legal landscape it needs to reason like a Swiss family lawyer.

**Why val_009**: 10 of 14 gold are in Stage A's top-2000, all are textbook Swiss family-law provisions (Art. 285 ZGB, Art. 277 ZGB, Art. 286 ZGB, Art. 291 ZGB, Art. 292 ZGB, Art. 100 BGG) — but v4's LLM rejected all 10 with confidence 0.10-0.15.

**Architecture (two passes)**:
1. **Pass 1** (one LLM call): from the query + aspects, ask Qwen3 to enumerate the canonical Swiss legal landscape per aspect — foundational statutes, BGE precedents, procedural rules, adjacent provisions.
2. **Pass 2** (per-candidate): inject the Pass-1 landscape into the prompt. Ask the LLM whether the candidate fits the landscape.

**Generalisable**: nothing is val_009-specific. Same code works for any query — just change `QID`. The Pass-1 landscape is LLM-generated at runtime, so no hardcoded lists.


## Phase 0 — Setup


In [ ]:
import os, sys, subprocess, json, time, gc, io, re
from pathlib import Path
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([
        "pip", "install", "-q", "-U",
        "vllm>=0.9.1", "transformers>=4.51.0", "pandas==2.2.3",
        "pyarrow==16.1.0", "numpy==1.26.4", "tqdm",
    ], check=True)


## Phase 1 — Paths and config

Single-query mode: set `QID` to any val query id. Default `val_009` (worst v4 F1).


In [ ]:
DRIVE_ROOT  = Path("/content/drive/MyDrive/swiss_law")
STAGE_B_IN  = DRIVE_ROOT / "research" / "stage_b_input" / "stage_b_input.parquet"
VAL_CSV     = DRIVE_ROOT / "data"     / "val.csv"
ALL_TARGETS = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "all_targets.json"
GOLD_SETS   = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "gold_doc_sets.json"
VAL_ASPECTS = DRIVE_ROOT / "research" / "concept_embedding_path" / "multi_aspect" / "val_aspects.parquet"

OUT_DIR = DRIVE_ROOT / "research" / "stage_b_grounded_llm" / "v5_single_query"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === Single-query knobs ===
QID          = "val_009"
TOP_K        = 2000
LLM_MODEL    = "Qwen/Qwen3-32B-AWQ"

# Pass 1 budget: legal landscape generation is one long reasoning call
PASS1_MAX_TOKENS    = 4096
PASS1_MAX_MODEL_LEN = 8192

# Pass 2 budget: per-candidate judgments, smaller reasoning trace
PASS2_MAX_TOKENS    = 1024
PASS2_MAX_MODEL_LEN = 8192

# Binding rules (same as v4)
AUTO_RULE = "article_match AND (co_citation_count >= 30 OR code_in_target)"
LLM_RULE  = "(article_match OR cc >= 5 OR concept_cosine >= 0.55) AND NOT AUTO"

print(f"QID: {QID}")
print(f"TOP_K: {TOP_K}")
print(f"LLM_MODEL: {LLM_MODEL}")
print(f"Output dir: {OUT_DIR}")
print("\nVerifying inputs:")
for p in [STAGE_B_IN, VAL_CSV, ALL_TARGETS, GOLD_SETS, VAL_ASPECTS]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")


## Phase 2 — Load query, aspects, candidates, gold


In [ ]:
import pandas as pd

# Query text
val_df = pd.read_csv(VAL_CSV)
query_text = str(val_df[val_df.query_id == QID].iloc[0]["query"])
print(f"=== {QID} QUERY ({len(query_text)} chars) ===")
print(query_text[:1500] + ("..." if len(query_text) > 1500 else ""))

# Aspects
asp_df = pd.read_parquet(VAL_ASPECTS)
aspects = list(asp_df[asp_df.query_id == QID].iloc[0]["aspects"])
print(f"\n=== ASPECTS ({len(aspects)}) ===")
for a in aspects:
    print(f"  {a.get('id'):<4} (weight={float(a.get('weight',0)):.2f}): {a.get('label')}")
    if 'concepts_en' in a: print(f"         concepts_en: {list(a['concepts_en'])}")
    if 'terms_de'    in a: print(f"         terms_de:    {list(a['terms_de'])}")

# Gold and candidates
gold_sets = json.load(open(GOLD_SETS, encoding="utf-8"))
gold_dids = set(gold_sets.get(QID, []))
total_gold = len(gold_dids)
print(f"\nGold dids total: {total_gold}")

sb_all = pd.read_parquet(STAGE_B_IN)
sb = (sb_all[sb_all.qid == QID]
        .sort_values("stage_a_rank")
        .head(TOP_K)
        .reset_index(drop=True))
gold_in_topk = int(sb.is_gold.sum())
print(f"In top-{TOP_K}: {gold_in_topk} ({gold_in_topk/max(1,total_gold):.1%} pool recall)")

# Tier assignment (same rules as v4)
sb["tier"] = "drop"
_auto = sb.article_match & ((sb.co_citation_count >= 30) | sb.code_in_target)
sb.loc[_auto, "tier"] = "auto"
_mid = (~_auto) & (sb.article_match | (sb.co_citation_count >= 5) | (sb.concept_cosine_score >= 0.55))
sb.loc[_mid, "tier"] = "llm"

print(f"\nTier breakdown:")
print(sb.groupby(["tier", "is_gold"]).size().unstack(fill_value=0).reindex(["auto","llm","drop"], fill_value=0))
print(f"\nLLM-tier prompts to generate: {(sb.tier=='llm').sum():,}")


## Phase 3 — Build the Pass-1 prompt (Swiss legal landscape)

This is the single most important new component. We ask Qwen3 in thinking mode to act as a Swiss-law expert and enumerate, per aspect, the canonical legal landscape: foundational statutes, BGE precedents, procedural rules, adjacent provisions.

The output of this single call becomes context for every Pass-2 per-candidate decision.


In [ ]:
PASS1_PROMPT = """You are a senior Swiss attorney with deep practical knowledge of every branch of Swiss federal and cantonal law (ZGB, OR, StGB, StPO, ZPO, BGG, SchKG, BV, EMRK, and the entire BGE jurisprudence). A client has come to you with the legal question below. Before evaluating any specific authorities, you need to map out the canonical Swiss legal landscape that a competent Swiss attorney would consult in writing an opinion on this question.

LEGAL QUESTION FROM THE CLIENT:
{question}

THE QUESTION'S LEGAL ASPECTS (already factored by an analyst — each aspect is one sub-domain you must map):
{aspects_block}

YOUR TASK:
For EACH aspect, enumerate the canonical Swiss legal authorities a competent Swiss lawyer would cite in their memorandum. Be exhaustive and specific — include article numbers and BGE references wherever you can.

For each aspect, produce these categories (where applicable):
1. FOUNDATIONAL STATUTES — the article(s) that directly establish the rule (e.g., for child support: Art. 285 ZGB)
2. CONSTITUTIONAL OR TREATY PROVISIONS — only if fundamental rights are at stake (BV, EMRK)
3. PROCEDURAL RULES — articles routinely cited in cases of this type (e.g., Art. 100 BGG for any Federal Tribunal appeal)
4. LEADING BGE PRECEDENTS — specific BGE numbers and the doctrines they establish
5. ADJACENT PROVISIONS — articles regularly co-cited alongside the foundational rule (e.g., Art. 277 ZGB + 285 ZGB + 286 ZGB are always cited together in child maintenance cases)
6. CROSS-ASPECT CONNECTIONS — note any place where one aspect's law interacts with another's

Be thorough — Swiss lawyers cite generously: every relevant article in the same statutory neighborhood, every related precedent, and every procedural rule that touches the case.

OUTPUT STRICT JSON (no prose before or after, begin with `{{`):
{{
  "a1": {{
    "aspect_label": "<copy the label from the input>",
    "foundational_statutes": [{{"cit": "Art. 285 ZGB", "role": "child support measurement based on needs and parental capacity"}}, ...],
    "constitutional_provisions": [{{"cit": "Art. 11 BV", "role": "..."}}, ...],
    "procedural_rules": [...],
    "leading_precedents": [{{"cit": "BGE 137 III 118", "doctrine": "hypothetical income may be imputed to incarcerated parents"}}, ...],
    "adjacent_provisions": [{{"cit": "Art. 286 ZGB", "role": "modification and indexing"}}, ...],
    "cross_aspect_connections": "..."
  }},
  "a2": {{...}},
  "a3": {{...}},
  "a4": {{...}}
}}
"""

def format_aspects_block(aspects):
    parts = []
    for a in aspects:
        aid = a.get("id", "")
        lbl = a.get("label", "")
        w   = float(a.get("weight", 0))
        terms_en = ", ".join(list(a.get("concepts_en", [])))
        terms_de = ", ".join(list(a.get("terms_de", [])))
        terms_fr = ", ".join(list(a.get("terms_fr", [])))
        terms_it = ", ".join(list(a.get("terms_it", [])))
        parts.append(
            f"  Aspect {aid} (priority weight={w:.2f}): {lbl}\n"
            f"    English concepts: {terms_en}\n"
            f"    German terms:     {terms_de}\n"
            f"    French terms:     {terms_fr}\n"
            f"    Italian terms:    {terms_it}"
        )
    return "\n".join(parts)

pass1_prompt = PASS1_PROMPT.format(
    question=query_text,
    aspects_block=format_aspects_block(aspects),
)
print(f"Pass-1 prompt length: {len(pass1_prompt):,} chars")
print(f"--- first 1200 chars ---")
print(pass1_prompt[:1200])


## Phase 4 — Run Pass 1 (generate Swiss legal landscape)

One LLM call, thinking enabled. Qwen3 thinking-mode sampling: `T=0.6, top_p=0.95, top_k=20, min_p=0`.


In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading {LLM_MODEL} on vLLM (thinking mode for Pass 1) ...")
llm = LLM(
    model=LLM_MODEL,
    dtype="bfloat16",
    gpu_memory_utilization=0.60,
    max_model_len=PASS1_MAX_MODEL_LEN,
    enforce_eager=False,
)

sp_pass1 = SamplingParams(
    temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
    max_tokens=PASS1_MAX_TOKENS,
    seed=42,
)

print(f"\nGenerating Pass-1 legal landscape (thinking ON) ...")
t0 = time.time()
out = llm.chat([[{"role": "user", "content": pass1_prompt}]], sampling_params=sp_pass1)
dt = time.time() - t0
pass1_raw = out[0].outputs[0].text
print(f"Done in {dt:.1f}s. Output length: {len(pass1_raw)} chars")

print("\n--- Pass 1 RAW OUTPUT (last 3000 chars; reasoning trace + JSON) ---")
print(pass1_raw[-3000:])


## Phase 5 — Parse Pass-1 landscape into structured form


In [ ]:
def parse_landscape(raw):
    s = raw.strip()
    # Strip thinking trace
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

landscape = parse_landscape(pass1_raw)
assert landscape is not None, "Failed to parse Pass-1 JSON; inspect pass1_raw"
print(f"Parsed landscape for {len(landscape)} aspects: {list(landscape.keys())}")

for aid, asp in landscape.items():
    print(f"\n=== {aid}: {asp.get('aspect_label','')} ===")
    for cat in ("foundational_statutes", "constitutional_provisions",
                "procedural_rules", "leading_precedents", "adjacent_provisions"):
        items = asp.get(cat, [])
        if not items: continue
        print(f"  {cat}:")
        for it in items[:8]:
            cit = it.get("cit", "")
            role = it.get("role", "") or it.get("doctrine", "") or it.get("topic", "")
            print(f"    - {cit:<25}  {role[:90]}")
    if asp.get("cross_aspect_connections"):
        print(f"  cross_aspect: {asp['cross_aspect_connections'][:200]}")


## Phase 6 — Build the Pass-2 prompt (per-candidate judgment with landscape)

The Pass-1 landscape is rendered into a compact text block and injected into every Pass-2 prompt. The LLM is told: "you've already mapped the legal landscape; now decide whether THIS candidate fits it."

This is the structural fix for v4's failure mode. With the landscape present, the LLM can recognize Art. 285 ZGB as "the foundational rule from aspect a1" instead of "an unfamiliar German article about general parental duties.


In [ ]:
def render_landscape(landscape):
    """Compact text block: lists every authority by aspect."""
    chunks = []
    for aid in sorted(landscape.keys()):
        asp = landscape[aid]
        lbl = asp.get("aspect_label", "")
        chunks.append(f"== {aid}: {lbl} ==")
        for cat, header in [
            ("foundational_statutes",     "Foundational statutes"),
            ("constitutional_provisions", "Constitutional/treaty"),
            ("procedural_rules",          "Procedural rules"),
            ("leading_precedents",        "Leading BGE precedents"),
            ("adjacent_provisions",       "Adjacent provisions (routinely co-cited)"),
        ]:
            items = asp.get(cat, [])
            if not items: continue
            chunks.append(f"  {header}:")
            for it in items:
                cit  = it.get("cit", "")
                role = it.get("role", "") or it.get("doctrine", "") or it.get("topic", "")
                chunks.append(f"    - {cit:<25}  {role[:120]}")
        if asp.get("cross_aspect_connections"):
            chunks.append(f"  Cross-aspect: {asp['cross_aspect_connections'][:300]}")
    return "\n".join(chunks)

landscape_text = render_landscape(landscape)
print(f"Rendered landscape: {len(landscape_text)} chars")
print(f"\n--- LANDSCAPE TEXT (first 2500 chars) ---")
print(landscape_text[:2500])

PASS2_PROMPT = """You are a senior Swiss lawyer. You have already mapped the canonical Swiss legal landscape for this client question (below). Now decide whether the candidate source is one a competent Swiss attorney would cite in their written legal opinion.

LEGAL QUESTION:
{question}

CANONICAL SWISS LEGAL LANDSCAPE FOR THIS QUESTION (from your prior legal-research pass):
{landscape}

CANDIDATE SOURCE TO EVALUATE:
- Citation: {citation}
- Type: {family_label}
- Code/court context: {family_extra}
- Paragraph role: {role}
- Substantive content (original language):
---BEGIN SOURCE TEXT---
{text}
---END SOURCE TEXT---

RETRIEVAL EVIDENCE (objective, advisory):
- Query names this article: {article_match}
- Co-cited with {co_citation_count} other Swiss court paragraphs in this query's pool
- Concept-cosine to query aspects: {concept_cosine_score:.2f}
- Code matches legal area: {code_in_target}

HOW TO DECIDE:
1. Look at the CITATION first. Is it (exactly or by article number) in your landscape above — under any aspect's foundational, constitutional, procedural, precedential, or adjacent categories?
2. If YES — KEEP. The candidate is a canonical authority for one of the question's aspects.
3. If the citation isn't literally in your landscape, look at the SOURCE TEXT. Does it state a rule that matches one of the aspects' legal subject matter (e.g., child-support measurement, security for future maintenance, capitalization, appeal procedure)?
4. If YES — KEEP, mark legal_role as "adjacent_provision" and note which aspect it serves.
5. REJECT only if the citation is NOT in your landscape AND the text is on a clearly unrelated legal topic (e.g., a tax provision in a maintenance case, a criminal-procedure rule in a civil-law question).

BE GENEROUS: Swiss lawyers cite every nearby article (e.g., Art. 277, 285, 286 ZGB all together in any child-maintenance opinion). When in doubt, KEEP — the cost of missing a foundational statute is high.

OUTPUT STRICT JSON (no prose):
{{"keep": true|false, "confidence": <0.0-1.0>, "matched_aspect": "<a1|a2|a3|a4|none>", "legal_role": "<foundational_statute|constitutional|procedural_rule|leading_precedent|adjacent_provision|off_topic>", "reasoning": "<one sentence>"}}
"""

def family_label(r):
    return "Swiss court precedent paragraph" if r.family == "court" else "Swiss statutory provision"

def family_extra(r):
    if r.family == "court":
        return f"Court base: {r.court_base!s}. Chamber: {r.chamber!s}."
    else:
        return f"Code: {r.law_code!s}. Law: {r.law_title!s}."

def build_pass2_prompt(r, landscape_text):
    return PASS2_PROMPT.format(
        question=query_text[:1500],
        landscape=landscape_text,
        citation=r.citation,
        family_label=family_label(r),
        family_extra=family_extra(r),
        role=r.role or "(unknown)",
        article_match=str(bool(r.article_match)).lower(),
        co_citation_count=int(r.co_citation_count),
        concept_cosine_score=float(r.concept_cosine_score),
        code_in_target=str(bool(r.code_in_target)).lower(),
        text=(r.text or "")[:1200].replace('"', "'"),
    )

sb_llm = sb[sb.tier == "llm"].reset_index(drop=True)
pass2_prompts = [build_pass2_prompt(r, landscape_text) for r in sb_llm.itertuples()]
print(f"\nBuilt {len(pass2_prompts):,} Pass-2 prompts.")
print(f"Approx prompt length: mean={int(sum(len(p) for p in pass2_prompts)/max(1,len(pass2_prompts))):,}  "
      f"max={max((len(p) for p in pass2_prompts), default=0):,}")
print(f"\n--- Sample Pass-2 prompt (first 1500 chars) ---")
print(pass2_prompts[0][:1500])


## Phase 7 — Run Pass 2

vLLM is already loaded. Re-use the same instance with a smaller `max_tokens` budget for the per-candidate reasoning.


In [ ]:
sp_pass2 = SamplingParams(
    temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
    max_tokens=PASS2_MAX_TOKENS,
    seed=42,
)

print(f"Running Pass 2 on {len(pass2_prompts):,} candidates (thinking ON) ...")
t0 = time.time()
messages = [[{"role": "user", "content": p}] for p in pass2_prompts]
outs2 = llm.chat(messages, sampling_params=sp_pass2)
dt = time.time() - t0
raw_texts = [o.outputs[0].text for o in outs2]
print(f"Done in {dt/60:.1f} min  ({len(pass2_prompts)/max(1,dt):.1f} candidates/sec)")

# Free LLM
del llm
gc.collect()
import torch; torch.cuda.empty_cache()

print("\n--- 2 sample Pass-2 outputs ---")
for i in (0, len(raw_texts)//2):
    print(f"\n[candidate {i}, len={len(raw_texts[i])}]")
    print(raw_texts[i][:500])


## Phase 8 — Parse Pass-2 outputs, combine AUTO + LLM tier


In [ ]:
def parse_pass2(raw):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

CONF_FLOOR = 0.30

# AUTO tier: forced keep
auto_rows = [{
    "did": r.did, "is_gold": bool(r.is_gold), "tier": "auto",
    "stage_a_rank": int(r.stage_a_rank), "citation": r.citation, "family": r.family,
    "keep_llm": True, "confidence": 1.0,
    "matched_aspect": "", "legal_role": "auto_dossier", "reasoning": "strong dossier signal",
    "keep_final": True, "parse_ok": True, "raw_llm": "",
} for r in sb[sb.tier == "auto"].itertuples()]

# LLM tier: parse + apply confidence floor
llm_rows = []
for r, raw in zip(sb_llm.itertuples(), raw_texts):
    parsed = parse_pass2(raw) or {}
    keep_llm = bool(parsed.get("keep", False))
    conf = float(parsed.get("confidence", 0.0) or 0.0)
    matched_aspect = str(parsed.get("matched_aspect", "") or "")
    legal_role = str(parsed.get("legal_role", "") or "")
    reasoning = str(parsed.get("reasoning", "") or "")
    keep_final = keep_llm and (conf >= CONF_FLOOR)
    llm_rows.append({
        "did": r.did, "is_gold": bool(r.is_gold), "tier": "llm",
        "stage_a_rank": int(r.stage_a_rank), "citation": r.citation, "family": r.family,
        "keep_llm": keep_llm, "confidence": conf,
        "matched_aspect": matched_aspect[:8], "legal_role": legal_role[:60],
        "reasoning": reasoning[:300],
        "keep_final": keep_final, "parse_ok": bool(parsed), "raw_llm": raw[:1500],
    })

out_df = pd.DataFrame(auto_rows + llm_rows)
print(f"v5 diagnostics for {QID}:")
print(f"  AUTO tier kept (no LLM):  {(out_df.tier=='auto').sum()}")
print(f"  LLM tier total:           {(out_df.tier=='llm').sum()}")
if (out_df.tier=='llm').sum() > 0:
    sub = out_df[out_df.tier == "llm"]
    print(f"    parse_ok:               {sub.parse_ok.mean():.1%}")
    print(f"    keep_llm=true:          {sub.keep_llm.mean():.1%}")
    print(f"    conf >= {CONF_FLOOR}:           {(sub.confidence >= CONF_FLOOR).mean():.1%}")
    print(f"    LLM-tier kept_final:    {sub.keep_final.mean():.1%}")
    print(f"  legal_role distribution:")
    print(out_df.legal_role.value_counts().head(10).to_string())
print(f"  TOTAL kept_final:         {out_df.keep_final.sum()}")


## Phase 9 — F1 for val_009 vs v4 baseline


In [ ]:
def f1(p, r):
    return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)

picks = set(out_df[out_df.keep_final].did)
correct = picks & gold_dids
p = len(correct) / max(1, len(picks))
r = len(correct) / max(1, total_gold)
f1_no_cap = f1(p, r)

print(f"=== v5 RESULTS for {QID} ===")
print(f"  Gold total:           {total_gold}")
print(f"  Gold in top-{TOP_K}:      {gold_in_topk}")
print(f"  v5 picks:             {len(picks)}")
print(f"  v5 correct:           {len(correct)}")
print(f"  P={p:.3f}  R={r:.3f}  F1={f1_no_cap:.3f}")
print(f"\n  v4 baseline for {QID}: P=0.000  R=0.000  F1=0.000")

# K-capped: sort kept by confidence, take top-K_gold
kept_sorted = out_df[out_df.keep_final].sort_values("confidence", ascending=False)
top_K = kept_sorted.head(total_gold)
picks_kcap = set(top_K.did)
correct_kcap = picks_kcap & gold_dids
p_k = len(correct_kcap) / max(1, len(picks_kcap))
r_k = len(correct_kcap) / max(1, total_gold)
print(f"\n  K-capped (K={total_gold}, by confidence):")
print(f"    P={p_k:.3f}  R={r_k:.3f}  F1={f1(p_k, r_k):.3f}")

# Per-aspect breakdown of LLM-kept candidates
print(f"\n=== Per-aspect breakdown (LLM-tier kept_final) ===")
llm_kept = out_df[(out_df.tier == "llm") & (out_df.keep_final)]
for a in ("a1", "a2", "a3", "a4"):
    sub = llm_kept[llm_kept.matched_aspect == a]
    if len(sub) == 0: continue
    n, g = len(sub), int(sub.is_gold.sum())
    print(f"  {a}: {n} kept, {g} gold ({g/n:.0%})")


## Phase 10 — What did v5 catch that v4 missed?


In [ ]:
# Gold survival
gold_rows = out_df[out_df.did.isin(gold_dids)].sort_values("stage_a_rank")
print(f"=== All 10 gold candidates (v5 verdict) ===")
print(f"{'rank':>4}  {'cit':<25}  {'tier':<5}  {'keep':>5}  {'conf':>5}  {'aspect':>6}  {'role':<22}  {'kept':>5}")
for _, r in gold_rows.iterrows():
    print(f"  {r.stage_a_rank:>4}  {r.citation[:25]:<25}  {r.tier:<5}  "
          f"{str(r.keep_llm):>5}  {r.confidence:>5.2f}  "
          f"{r.matched_aspect[:6]:>6}  {r.legal_role[:22]:<22}  {str(r.keep_final):>5}")

# Show 5 example reasonings for kept gold
print(f"\n=== Sample reasonings for kept gold ===")
for _, r in gold_rows[gold_rows.keep_final].head(5).iterrows():
    print(f"\n--- {r.citation}  (rank {r.stage_a_rank}, aspect {r.matched_aspect}) ---")
    print(f"  {r.reasoning}")

# Show 3 example reasonings for missed gold (kept_final=False)
print(f"\n=== Sample reasonings for MISSED gold ===")
for _, r in gold_rows[~gold_rows.keep_final].head(3).iterrows():
    print(f"\n--- {r.citation}  (rank {r.stage_a_rank}) ---")
    print(f"  conf={r.confidence:.2f}  matched_aspect={r.matched_aspect}  legal_role={r.legal_role}")
    print(f"  {r.reasoning}")


## Phase 11 — Save outputs


In [ ]:
out_df.to_parquet(OUT_DIR / f"{QID}_v5_raw.parquet", index=False)

# Survivors
survivors = out_df[out_df.keep_final][["did","citation","family","tier","matched_aspect","legal_role","confidence","is_gold","reasoning"]]
survivors.to_parquet(OUT_DIR / f"{QID}_v5_survivors.parquet", index=False)

# Landscape
with open(OUT_DIR / f"{QID}_v5_landscape.json", "w", encoding="utf-8") as f:
    json.dump(landscape, f, indent=2, ensure_ascii=False)

# Metrics
metrics = {
    "qid": QID,
    "config": {"TOP_K": TOP_K, "LLM_MODEL": LLM_MODEL,
               "PASS1_MAX_TOKENS": PASS1_MAX_TOKENS, "PASS2_MAX_TOKENS": PASS2_MAX_TOKENS,
               "CONF_FLOOR": CONF_FLOOR},
    "tiers": {
        "auto": int((sb.tier == "auto").sum()),
        "llm":  int((sb.tier == "llm").sum()),
        "drop": int((sb.tier == "drop").sum()),
    },
    "gold_total": total_gold,
    "gold_in_topk": gold_in_topk,
    "picks": len(picks),
    "correct": len(correct),
    "P_no_cap": p, "R_no_cap": r, "F1_no_cap": f1_no_cap,
    "P_k_cap": p_k, "R_k_cap": r_k, "F1_k_cap": f1(p_k, r_k),
    "v4_baseline_F1": 0.000,
}
with open(OUT_DIR / f"{QID}_v5_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved:")
print(f"  {OUT_DIR / (QID + '_v5_raw.parquet')}")
print(f"  {OUT_DIR / (QID + '_v5_survivors.parquet')}")
print(f"  {OUT_DIR / (QID + '_v5_landscape.json')}")
print(f"  {OUT_DIR / (QID + '_v5_metrics.json')}")
print(f"\nFinal F1 for {QID}: {f1_no_cap:.3f}  (v4 was 0.000)")
